# Supplementary Figure 4 — single-feature models WITHOUT noise correction

Same models, folds, features and inference as Figure 3a/b. The only difference is
that predictivity is *not* divided by each condition's split-half reliability, so
these bars are what the models actually achieve against noisy image means, and the
ceiling is the reliability rather than 1. The unique-contribution analysis is not
repeated: the point here is the scale of the raw scores, and it is the same point
in both model families.

Nothing is recomputed. The correction is one positive constant per condition, so
multiplying the cached corrected values back by it recovers the raw scores and
their fold SDs exactly:

| task | preview | no preview |
|---|---|---|
| face detection | x .739 | x .859 |
| moving window | x .824 | x .862 |

**The p-values.** `p>0` is *identical* with and without the correction: it scales
the observed statistic and its permutation null by the same constant, so the
comparison is unchanged. The preview-minus-no-preview `delta` is different — the
two conditions scale by different constants, so the uncorrected delta strictly has
its own null — and the p carried over here is the corrected one, marked `p*`. No
feature's delta survives correction for multiple comparisons under either version,
so the distinction changes no conclusion.

Standardised betas are not repeated either: they are never noise-corrected, so the
values in Figure 3 already are the uncorrected ones.

In [ ]:
import sys
from pathlib import Path

# The notebooks live next to the code, so the package is importable without
# installing anything. Nothing outside this release is referenced.
sys.path.insert(0, str(Path.cwd().parent / "code"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from facescenes import config as cfg
from facescenes import data as D
from facescenes import behaviour as B
from facescenes import encoding as E
from facescenes import onset_gaze as G
from facescenes import plotting as P
from facescenes import stats as S

P.use_style()
print("stimuli    :", len(D.stimulus_names()))
for t in cfg.TASKS:
    print(f"{t:14s}: {len(D.participant_ids(t))} participants, "
          f"{len(D.analysable_images(t))} images")

In [ ]:
import run_encoding_analyses as RUN

RES = {t: RUN.analyse(t, n_perm=5000) for t in cfg.TASKS}


def raw_panel(task, fname):
    res = RES[task]
    labels = cfg.FEATURE_LABELS
    rel = cfg.RELIABILITY[task]
    # Undo the correction: predictivity and its fold SD were both divided by rel.
    prev = res["single_preview"] * rel["preview"]
    nop = res["single_no_preview"] * rel["no_preview"]
    sd_p = res["single_sd_preview"] * rel["preview"]
    sd_n = res["single_sd_no_preview"] * rel["no_preview"]
    x = np.arange(len(labels))

    fig, ax = plt.subplots(figsize=(14, 9))
    ax.bar(x - .2, prev, .4, color=cfg.C_PREVIEW, label="Preview")
    ax.errorbar(x - .2, prev, yerr=P.one_sided_errorbar(prev, sd_p), fmt="None",
                color="k", lw=1.2)
    ax.bar(x + .2, nop, .4, color=cfg.C_NO_PREVIEW, label="No preview")
    ax.errorbar(x + .2, nop, yerr=P.one_sided_errorbar(nop, sd_n), fmt="None",
                color="k", lw=1.2)
    ax.axhline(0, color="k", lw=.8)

    lo = min(0, np.min(np.minimum(prev - sd_p, nop - sd_n)))
    hi = np.max(np.maximum(prev + sd_p, nop + sd_n)) * 1.18
    ticks, (t0, t1) = P.zero_anchored_ticks(lo, hi)
    ax.set_ylim(t0, t1); ax.set_yticks(ticks)
    ax.set_yticklabels([f"{v:g}" for v in ticks], fontsize=P.FONT - 6)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=P.FONT - 6)
    # The one uncorrected predictivity axis in the release.
    ax.set_ylabel(P.RAW_PREDICTIVITY_LABEL, fontsize=P.FONT - 4)
    P.title(ax, cfg.TASK_LABEL[task], fontsize=P.FONT - 2, pad=14)
    if task == "facedetection":                       # one legend serves both panels
        ax.legend(fontsize=P.FONT - 8, frameon=False, loc="upper left")
    P.clean(ax)
    # Same asterisks as Figure 3a/b: p>0 is unchanged by the correction, so
    # one-sided, Bonferroni x 20, a single level (smallest corrected p is .004).
    n_fam = len(cfg.FEATURE_NAMES) * 2
    for xs, vals, sds, ps in ((x - .2, prev, sd_p, res["p_single_preview"]),
                              (x + .2, nop, sd_n, res["p_single_no_preview"])):
        for xi, v, s, p in zip(xs, vals, sds, ps):
            if p * n_fam < .05:
                ax.text(xi, max(v, 0) + s + .01 * (t1 - t0), "*", ha="center",
                        va="bottom", fontsize=P.FONT - 8)
    pp, pn = res["p_single_preview"], res["p_single_no_preview"]
    src = pd.DataFrame({
        "feature": labels,
        "preview_r2_uncorrected": prev, "preview_sd_across_folds": sd_p,
        "preview_p": pp, "preview_p_bonferroni": np.minimum(1, pp * n_fam),
        "no_preview_r2_uncorrected": nop, "no_preview_sd_across_folds": sd_n,
        "no_preview_p": pn, "no_preview_p_bonferroni": np.minimum(1, pn * n_fam)})
    P.save(fig, fname, src); plt.show()

    n_p0, n_delta = len(labels) * 2, len(labels)
    print(f"{cfg.TASK_LABEL[task]}  —  single-feature predictivity, NOT noise corrected")
    print(f"  {'feature':14s} {'preview':>16s} {'p>0':>7s} {'bonf':>6s}  "
          f"{'no preview':>16s} {'p>0':>7s} {'bonf':>6s}  "
          f"{'delta':>8s} {'p*':>7s} {'bonf':>6s}")
    for k, lab in enumerate(labels):
        p0p, p0n = res["p_single_preview"][k], res["p_single_no_preview"][k]
        p_d = res["p_delta_single"][k]
        print(f"  {lab:14s} {prev[k]:+8.3f}+/-{sd_p[k]:.3f} "
              f"{p0p:7.4f} {min(1, p0p*n_p0):6.3f}  "
              f"{nop[k]:+8.3f}+/-{sd_n[k]:.3f} "
              f"{p0n:7.4f} {min(1, p0n*n_p0):6.3f}  "
              f"{prev[k]-nop[k]:+8.3f} {p_d:7.4f} {min(1, p_d*n_delta):6.3f}")
    print(f"  ceiling here is the reliability, not 1: "
          f"preview {rel['preview']:.3f}, no preview {rel['no_preview']:.3f}")
    print("  p>0 identical to the corrected analysis; p* carried over from it")


raw_panel("facedetection", "suppfig4a_single_raw_face_detection.png")

In [ ]:
raw_panel("movingwindow", "suppfig4b_single_raw_moving_window.png")